# Deplección y Ciclo de Combustible
## Quemado, evolución de isótopos y gestión del ciclo en el NuScale

**Máster THN — UPM | Práctica 3**

---

En este notebook estudiamos la **evolución del combustible a lo largo del ciclo de operación**. El reactor NuScale opera con ciclos de ~24 meses, durante los cuales el combustible se va "quemando" (deplección), lo que modifica la reactividad del núcleo y requiere compensación continua.

### Contenidos

1. Concepto de quemado (burnup) y unidades
2. Evolución de isótopos clave a lo largo del ciclo
3. Evolución de k_eff y gestión de la reactividad
4. Venenos neutrónico: Xe-135 y Sm-149
5. Barras de control y boro soluble
6. Estrategia de recarga parcial


---
## 0. Configuración


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
})

# Constantes físicas
N_A = 6.022e23   # Número de Avogadro
MWD_TO_J = 86.4e9  # 1 MWd = 86.4e9 J

---
## 1. Quemado (Burnup)

El **quemado** (o burnup) cuantifica la cantidad de energía producida por unidad de masa de metal pesado inicial (HM) en el combustible:

$$\text{Burnup} \; [\text{MWd/tU}] = \frac{\text{Energía total producida [MWd]}}{\text{Masa inicial de uranio [tU]}}$$

### Valores típicos

| Reactor | Burnup promedio [MWd/tU] | Burnup máx. pin |
|---|---|---|
| PWR estándar (ciclo 18 meses) | 40,000 – 50,000 | ~60,000 |
| NuScale (ciclo 24 meses) | ~30,000 – 40,000 | ~45,000 |
| AP1000 (ciclo 24 meses) | ~45,000 | ~62,000 |

### Relación con el tiempo

Para potencia constante:
$$B(t) = \frac{P_{específica} \cdot t}{1000}$$

donde $P_{específica}$ es la potencia en MW/tU.


In [ ]:
# Parámetros del ciclo NuScale
Q_MWt = 160.0            # Potencia térmica [MWt]
M_U_total_tU = 13.7      # Masa total de uranio en el núcleo [tU] (aprox. 37 FA × 370 kg/FA)
P_esp = Q_MWt / M_U_total_tU  # Potencia específica [MW/tU]

t_ciclo_dias = 730        # Duración del ciclo [días] ≈ 24 meses
t = np.linspace(0, t_ciclo_dias, 200)  # Tiempo [días]

# Burnup en función del tiempo (potencia constante)
BU = P_esp * t  # MWd/tU

print(f"Potencia específica: {P_esp:.1f} MW/tU")
print(f"Burnup al EOC ({t_ciclo_dias} días): {BU[-1]:.0f} MWd/tU")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t, BU/1000, 'steelblue', linewidth=2.5)
ax.axvline(t_ciclo_dias/3, color='orange', ls='--', alpha=0.7, label='1/3 ciclo')
ax.axvline(2*t_ciclo_dias/3, color='green', ls='--', alpha=0.7, label='2/3 ciclo')
ax.set_xlabel('Tiempo [días]', fontsize=12)
ax.set_ylabel('Burnup [GWd/tU]', fontsize=12)
ax.set_title(f'Evolución del quemado — NuScale SMR (ciclo {t_ciclo_dias} días)',
             fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## 2. Evolución de isótopos

### 2.1 Ecuaciones de Bateman

La evolución de los isótopos en el combustible nuclear se rige por las **ecuaciones de Bateman** (o ecuaciones de evolución de la quema):

$$\frac{dN_i}{dt} = \sum_j \gamma_{ji} \sigma_{f,j} \phi N_j + \sum_k \lambda_k \beta_{ki} N_k - (\sigma_{a,i} \phi + \lambda_i) N_i$$

donde:
- $N_i$: concentración del isótopo $i$ [átomos/cm³]
- $\sigma_{f,j}$: sección eficaz de fisión del isótopo $j$ [cm²]
- $\phi$: flujo neutrónico [n/(cm²·s)]
- $\lambda_i$: constante de decaimiento radioactivo [s⁻¹]
- $\gamma_{ji}$: rendimiento de fisión del isótopo $i$ a partir de $j$


In [ ]:
# Simulación simplificada de evolución de isótopos principales
# Datos nucleares aproximados (para fines educativos)

# Flujo neutrónico promedio [n/(cm²·s)]
phi = 3.5e13  # típico para PWR de media potencia

# Secciones eficaces [barn = 1e-24 cm²]
# U-235
sigma_f_U235 = 530e-24   # cm²
sigma_a_U235 = 650e-24   # cm²
# U-238
sigma_a_U238 = 2.7e-24   # cm²
sigma_c_U238 = 2.7e-24   # captura → Pu-239
# Pu-239
sigma_f_Pu239 = 750e-24  # cm²
sigma_a_Pu239 = 1010e-24 # cm²

# Densidades iniciales [átomos/cm³] (enriquecimiento ~4.95%)
rho_UO2 = 10.96  # g/cm³
M_UO2 = 270.0    # g/mol
N_UO2 = rho_UO2 * N_A / M_UO2  # mol/cm³

enriq = 0.0495
N_U235_0 = enriq * N_UO2 * 1e-3        # átomos/cm³
N_U238_0 = (1-enriq) * N_UO2 * 1e-3   # átomos/cm³
N_Pu239_0 = 0.0

# Tiempo en segundos
t_s = t * 86400  # días a segundos

def ecuaciones_bateman(t, y):
    """Sistema de ecuaciones de Bateman simplificado para U235, U238, Pu239."""
    N_U235, N_U238, N_Pu239 = y
    
    # Decaimiento de U235: dN_U235/dt = -σ_a_U235 * φ * N_U235
    dU235 = -sigma_a_U235 * phi * N_U235
    
    # U238: captura neutrónica → Pu-239
    dU238 = -sigma_a_U238 * phi * N_U238
    
    # Pu-239: producción (captura en U238) - destrucción (absorción)
    # Simplificación: Np-239 decae rápido → Pu-239 directamente
    lambda_Pu239 = np.log(2) / (2.41e4 * 365.25 * 86400)  # T_1/2 = 24100 años
    dPu239 = sigma_c_U238 * phi * N_U238 - (sigma_a_Pu239 * phi + lambda_Pu239) * N_Pu239
    
    return [dU235, dU238, dPu239]

# Resolver sistema
sol = solve_ivp(
    ecuaciones_bateman,
    [t_s[0], t_s[-1]],
    [N_U235_0, N_U238_0, N_Pu239_0],
    t_eval=t_s,
    method='RK45',
    rtol=1e-6
)

N_U235 = sol.y[0] / N_U235_0  # Normalizado
N_U238 = sol.y[1] / N_U238_0
N_Pu239 = sol.y[2] / N_U235_0  # Relativo al U235 inicial

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(t, N_U235 * 100, 'steelblue', linewidth=2.5, label='U-235 (% del inicial)')
ax.plot(t, N_U238 * 100, 'gray', linewidth=2.5, ls='--', label='U-238 (% del inicial)')
ax2 = ax.twinx()
ax2.plot(t, N_Pu239 * 100, 'r-', linewidth=2.5, label='Pu-239 (% del U235 inicial)')
ax2.set_ylabel('Pu-239 (% del U235 inicial)', color='red', fontsize=11)
ax2.tick_params(axis='y', labelcolor='red')

ax.set_xlabel('Tiempo [días]', fontsize=12)
ax.set_ylabel('Concentración [% del inicial]', fontsize=11)
ax.set_title('Evolución de isótopos de actínidos — ciclo NuScale', fontweight='bold')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=10, loc='center right')
plt.tight_layout()
plt.show()

print(f"U-235 al EOC: {N_U235[-1]*100:.1f}% del inicial")
print(f"Pu-239 al EOC: {N_Pu239[-1]*100:.2f}% del U235 inicial")

---
## 3. Evolución de k_eff durante el ciclo

A medida que avanza el ciclo, el U-235 se consume y los productos de fisión se acumulan, reduciendo gradualmente $k_{eff}$. Para mantener la criticidad, se retiran las barras de control progresivamente.

El **exceso de reactividad** al inicio del ciclo (BOC) es:
$$\rho_{exc} = \frac{k_{eff,BOC} - 1}{k_{eff,BOC}}$$


In [ ]:
# Evolución de k_eff a lo largo del ciclo
# Contribuciones:
#   - Deplección del U235: reduce k
#   - Acumulación de Pu239: aumenta k (parcialmente compensa)
#   - Productos de fisión (Xe, Sm, etc.): reducen k
#   - Quema de venenos combustibles (IFBA, Gd): aumenta k al inicio

# k_eff calculado (simplificado)
keff_BOC = 1.035  # Exceso de reactividad al inicio
keff_EOC = 1.002  # Reactividad al final (barras retiradas)

# Evolución aproximada (decaimiento no lineal)
# Quema de venenos IFBA compensa al inicio:
keff_base = keff_BOC - (keff_BOC - keff_EOC) * (t / t_ciclo_dias)**0.9
# Quema de Gd produce un pico a ~100 días
keff_Gd_quema = 0.003 * np.exp(-t/120) * (1 - np.exp(-t/30))
keff_ciclo = keff_base + keff_Gd_quema

# Reactividad neta con barras de control (lo que el operador ve)
# Las barras se van retirando para compensar la deplección
# Se modela como una reactividad de compensación por barras
rho_barras = (keff_ciclo - 1.0) / keff_ciclo  # Reactividad excedente que absorben las barras

fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)

ax = axes[0]
ax.plot(t, keff_ciclo, 'steelblue', linewidth=2.5, label='k_eff (exceso total)')
ax.axhline(1.0, color='red', ls='--', linewidth=1.5, label='Criticidad (k=1)')
ax.fill_between(t, 1.0, keff_ciclo, alpha=0.2, color='orange',
                label='Exceso de reactividad a compensar')
ax.set_ylabel('k_eff [-]', fontsize=12)
ax.set_title('Evolución de k_eff durante el ciclo — NuScale', fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0.99, 1.045)

# Marcar etapas del ciclo
for label, pos, color in [('BOC', 0, 'green'), ('MOC', t_ciclo_dias/2, 'blue'),
                            ('EOC', t_ciclo_dias, 'red')]:
    ax.axvline(pos, color=color, ls=':', alpha=0.6)
    ax.text(pos+5, 1.042, label, color=color, fontsize=10, fontweight='bold')

ax2 = axes[1]
rho_pcm = rho_barras * 1e5
ax2.plot(t, rho_pcm, 'darkorange', linewidth=2.5)
ax2.fill_between(t, 0, rho_pcm, alpha=0.3, color='orange')
ax2.set_xlabel('Tiempo [días]', fontsize=12)
ax2.set_ylabel('Reactividad excedente [pcm]', fontsize=12)
ax2.set_title('Reactividad excedente compensada por barras de control',
               fontweight='bold')
ax2.axhline(0, color='k', ls='--', alpha=0.5)

plt.tight_layout()
plt.show()

print(f"Reactividad excedente al BOC: {(keff_ciclo[0]-1)/keff_ciclo[0]*1e5:.0f} pcm")
print(f"Quemado al EOC: {BU[-1]:.0f} MWd/tU")

---
## 4. Venenos neutrónico: Xe-135 y Sm-149

### 4.1 El xenón-135 (Xe-135)

El **Xe-135** es el veneno neutrónico más importante en los reactores PWR. Tiene la mayor sección eficaz de absorción de todos los elementos conocidos (~2.6 × 10⁶ barn).

Se produce principalmente por **decaimiento del I-135** (producto de fisión):
$$\text{Te-135} \xrightarrow{\beta} \text{I-135} \xrightarrow{\beta, T_{1/2}=6.6h} \text{Xe-135} \xrightarrow{\beta, T_{1/2}=9.2h} \text{Cs-135}$$

### 4.2 El samario-149 (Sm-149)

El **Sm-149** se produce por decaimiento del Pm-149:
$$\text{Nd-149} \xrightarrow{\beta} \text{Pm-149} \xrightarrow{\beta, T_{1/2}=53.1h} \text{Sm-149}$$

A diferencia del Xe-135, el Sm-149 es estable (no decae) y solo se elimina por absorción neutrónica.


In [ ]:
# Cinética del Xe-135 y Sm-149 durante arranque, operación y parada

# Datos nucleares
sigma_Xe = 2.6e6 * 1e-24  # cm² sección eficaz de absorción Xe-135
lambda_I  = np.log(2) / (6.59 * 3600)   # s⁻¹ decaimiento I-135
lambda_Xe = np.log(2) / (9.17 * 3600)   # s⁻¹ decaimiento Xe-135
lambda_Pm = np.log(2) / (53.1 * 3600)   # s⁻¹ decaimiento Pm-149

# Rendimientos de fisión
gamma_I  = 0.0639  # rendimiento directo I-135 (incluye Te-135 → I-135)
gamma_Xe = 0.00228  # rendimiento directo Xe-135 
gamma_Pm = 0.0113   # rendimiento Pm-149

# Flujo neutrónico y sección eficaz de fisión
Sigma_f = 0.30  # cm⁻¹ sección eficaz de fisión macroscópica

def xenon_steady_state():
    """Concentración de equilibrio de Xe-135 durante operación."""
    # Concentraciones de equilibrio (dN/dt = 0)
    I_eq = gamma_I * Sigma_f * phi / lambda_I
    Xe_eq = (gamma_Xe * Sigma_f * phi + lambda_I * I_eq) / (lambda_Xe + sigma_Xe * phi)
    return I_eq, Xe_eq

I_eq, Xe_eq = xenon_steady_state()

# Simulación temporal: arranque → operación → parada → xenon peak
# Tiempo en horas
t_h = np.linspace(0, 72, 1000)  # 72 horas
t_s2 = t_h * 3600

# Fases:
# 0-5h: arranque
# 5-30h: operación plena
# 30-72h: parada (phi=0)
t_start = 5.0   # hora de arranque a plena potencia
t_stop  = 30.0  # hora de parada

def phi_t(t_h):
    """Flujo en función del tiempo."""
    if t_h < t_start:
        return phi * t_h / t_start  # Rampa de subida
    elif t_h < t_stop:
        return phi
    else:
        return 0.0

def dXe_sistema(t, y):
    I, Xe = y
    ph = phi_t(t/3600)
    dI  = gamma_I * Sigma_f * ph - lambda_I * I
    dXe = gamma_Xe * Sigma_f * ph + lambda_I * I - (lambda_Xe + sigma_Xe * ph) * Xe
    return [dI, dXe]

sol_Xe = solve_ivp(dXe_sistema, [0, t_s2[-1]], [0, 0],
                   t_eval=t_s2, method='RK45', rtol=1e-8)

Xe = sol_Xe.y[1] / Xe_eq  # Normalizado al equilibrio

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(t_h, Xe, 'b-', linewidth=2.5, label='Xe-135 / Xe_equilibrio')
ax.axhline(1.0, color='steelblue', ls='--', alpha=0.6, label='Nivel de equilibrio')
ax.axvline(t_start, color='green', ls=':', linewidth=1.5)
ax.axvline(t_stop, color='red', ls=':', linewidth=1.5)
ax.text(t_start + 0.5, 1.5, 'Plena potencia', color='green', fontsize=10)
ax.text(t_stop + 0.5, 1.5, 'PARADA', color='red', fontsize=10, fontweight='bold')

# Anotación del pico de xenón
idx_max = np.argmax(Xe)
ax.annotate(f'Pico Xe: {Xe[idx_max]:.2f}× equilibrio\nt = {t_h[idx_max]:.1f}h',
            xy=(t_h[idx_max], Xe[idx_max]),
            xytext=(t_h[idx_max]+3, Xe[idx_max]+0.1),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=10, fontweight='bold')

ax.set_xlabel('Tiempo [horas]', fontsize=12)
ax.set_ylabel('Xe-135 (relativo al equilibrio)', fontsize=12)
ax.set_title('Transitorio de Xe-135: arranque → operación → parada\n("Xenon peak" o pozo de xenón)',
             fontweight='bold')
ax.legend(fontsize=10)

# Sombrear fases
ax.axvspan(0, t_start, alpha=0.1, color='yellow', label='Arranque')
ax.axvspan(t_start, t_stop, alpha=0.1, color='green')
ax.axvspan(t_stop, 72, alpha=0.1, color='red')

plt.tight_layout()
plt.show()

---
## 5. Gestión del ciclo: barras de control y recarga

### 5.1 Posición de las barras de control durante el ciclo

En el NuScale, las barras de control se usan para:
1. **Control de reactividad**: compensar la deplección del combustible
2. **Control de potencia**: ajustar el nivel de potencia
3. **Distribución axial**: achatar el perfil de potencia


In [ ]:
# Posición de barras vs tiempo del ciclo
# BOC: barras insertadas (compensan el exceso de reactividad)
# EOC: barras retiradas

t_ciclo = np.linspace(0, 730, 100)  # días

# Inserción de barras (0 = totalmente insertadas, 100% = totalmente retiradas)
# La extracción es progresiva con la deplección, más rápida al inicio
pos_barras = 100 * (1 - np.exp(-3 * t_ciclo / 730))
# Corrección: al inicio se queman los venenos Gd, lo que permite retirar barras más rápido
pos_barras_real = pos_barras * (1 + 0.1 * np.exp(-t_ciclo/50))
pos_barras_real = np.minimum(pos_barras_real, 100)

# Factor de potencia relativo
P_rel = 1.0 + 0.05 * np.sin(np.pi * t_ciclo / 730)  # Ligera variación

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax = axes[0]
ax.plot(t_ciclo, pos_barras_real, 'darkred', linewidth=2.5)
ax.fill_between(t_ciclo, 0, pos_barras_real, alpha=0.2, color='red')
ax.set_ylabel('Posición de barras [%]\n(0=insertadas, 100=retiradas)', fontsize=11)
ax.set_title('Gestión de barras de control durante el ciclo — NuScale', fontweight='bold')
ax.set_ylim(0, 110)

for label, pos, color in [('BOC', 0, 'green'), ('MOC', 365, 'blue'), ('EOC', 730, 'red')]:
    ax.axvline(pos, color=color, ls=':', alpha=0.7)
    ax.text(pos+5, 105, label, color=color, fontsize=10, fontweight='bold')

ax2 = axes[1]
ax2.plot(t_ciclo, keff_ciclo[:len(t_ciclo)], 'steelblue', linewidth=2.5, label='k_eff sin barras')
# k_eff efectivo con barras (debe ser ≈1.0)
rho_barras_ciclo = (keff_ciclo[:len(t_ciclo)] - 1) / keff_ciclo[:len(t_ciclo)]
keff_con_barras = np.ones_like(t_ciclo) + 0.001 * np.sin(np.pi * t_ciclo / 730 * 2)  # ~1.0
ax2.plot(t_ciclo, keff_con_barras, 'g--', linewidth=2.5, label='k_eff neto (operación crítica)')
ax2.axhline(1.0, color='red', ls=':', alpha=0.6, label='Criticidad')
ax2.set_xlabel('Tiempo [días]', fontsize=12)
ax2.set_ylabel('k_eff [-]', fontsize=12)
ax2.set_title('k_eff con y sin compensación por barras', fontweight='bold')
ax2.legend(fontsize=10)
ax2.set_ylim(0.995, 1.04)

plt.tight_layout()
plt.show()

### 5.2 Estrategia de recarga parcial (Out-In)

Al final de cada ciclo (~24 meses), se realiza una **recarga parcial**:

- Se extraen los ensamblajes más quemados (periferia)
- Se desplazan ensamblajes de media quema hacia la periferia
- Se insertan ensamblajes frescos en el centro

El NuScale utiliza una estrategia de recarga en 3 regiones (low-leakage loading pattern).


In [ ]:
import matplotlib.patches as mpatches

# Mapa del núcleo con regiones de quemado
CORE_MAP = [
    (0,2),(0,3),(0,4),
    (1,1),(1,2),(1,3),(1,4),(1,5),
    (2,0),(2,1),(2,2),(2,3),(2,4),(2,5),(2,6),
    (3,0),(3,1),(3,2),(3,3),(3,4),(3,5),(3,6),
    (4,0),(4,1),(4,2),(4,3),(4,4),(4,5),(4,6),
    (5,1),(5,2),(5,3),(5,4),(5,5),
    (6,2),(6,3),(6,4),
]

center = (3.0, 3.0)
distancias = [np.sqrt((r-center[0])**2 + (c-center[1])**2) for r, c in CORE_MAP]

# Asignar región según distancia al centro
# Región 1 (fresca, centro): d < 1.5
# Región 2 (media quema): 1.5 <= d < 2.8
# Región 3 (alta quema, periferia): d >= 2.8
regiones = []
for d in distancias:
    if d < 1.5:
        regiones.append(1)
    elif d < 2.8:
        regiones.append(2)
    else:
        regiones.append(3)

color_region = {1: '#2196F3', 2: '#FF9800', 3: '#F44336'}
label_region = {1: 'Región 1 (fresca)', 2: 'Región 2 (media quema)', 3: 'Región 3 (alta quema)'}
BU_region = {1: '~5,000', 2: '~20,000', 3: '~35,000'}

fig, ax = plt.subplots(1, 1, figsize=(8, 8))

for i, (row, col) in enumerate(CORE_MAP):
    reg = regiones[i]
    rect = mpatches.FancyBboxPatch(
        (col * 1.05, (6 - row) * 1.05), 0.95, 0.95,
        boxstyle="round,pad=0.03",
        facecolor=color_region[reg], edgecolor='white', linewidth=1.5, alpha=0.85
    )
    ax.add_patch(rect)
    ax.text(col*1.05+0.475, (6-row)*1.05+0.55, str(reg),
            ha='center', va='center', fontsize=9, color='white', fontweight='bold')
    ax.text(col*1.05+0.475, (6-row)*1.05+0.22, BU_region[reg],
            ha='center', va='center', fontsize=6, color='white')

# Leyenda
legend_patches = [
    mpatches.Patch(color=color_region[r], label=f'{label_region[r]}\n~{BU_region[r]} MWd/tU')
    for r in [1, 2, 3]
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9,
          title='Quemado al inicio del ciclo')

ax.set_xlim(-0.3, 7.6)
ax.set_ylim(-0.3, 7.6)
ax.set_aspect('equal')
ax.set_title('Mapa de carga — Estrategia Low-Leakage\nNuScale SMR (inicio de ciclo)',
             fontsize=12, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

n_r = {r: regiones.count(r) for r in [1, 2, 3]}
print(f"Distribución de ensamblajes por región:")
for r, n in n_r.items():
    print(f"  Región {r}: {n} FA ({n/37*100:.0f}% del núcleo)")

---
## 6. Resumen del ciclo completo


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Burnup
ax = axes[0, 0]
ax.plot(t, BU/1000, 'steelblue', linewidth=2.5)
ax.set_xlabel('Tiempo [días]'); ax.set_ylabel('Burnup [GWd/tU]')
ax.set_title('Quemado acumulado', fontweight='bold')

# 2. Evolución de isótopos
ax = axes[0, 1]
ax.plot(t, N_U235*100, 'b-', linewidth=2, label='U-235 [%]')
ax.plot(t, N_Pu239*100, 'r--', linewidth=2, label='Pu-239 [% del U235_0]')
ax.set_xlabel('Tiempo [días]'); ax.set_ylabel('Concentración [%]')
ax.set_title('Evolución de actínidos', fontweight='bold')
ax.legend(fontsize=9)

# 3. k_eff
ax = axes[1, 0]
ax.plot(t, keff_ciclo, 'darkorange', linewidth=2.5)
ax.axhline(1.0, color='red', ls='--', alpha=0.6)
ax.set_xlabel('Tiempo [días]'); ax.set_ylabel('k_eff [-]')
ax.set_title('Factor de multiplicación efectivo', fontweight='bold')
ax.set_ylim(0.99, 1.04)

# 4. Xenón
ax = axes[1, 1]
t_xe_h = np.linspace(0, 72, 1000)
ax.plot(t_xe_h[:len(Xe)], Xe, 'purple', linewidth=2)
ax.axhline(1.0, color='gray', ls='--', alpha=0.5, label='Equilibrio')
ax.axvline(t_stop, color='red', ls=':', alpha=0.7, label='Parada')
ax.set_xlabel('Tiempo [h]'); ax.set_ylabel('Xe-135 / Xe_eq [-]')
ax.set_title('Transitorio de Xe-135 (parada)', fontweight='bold')
ax.legend(fontsize=9)

plt.suptitle('Resumen del ciclo de operación — NuScale SMR',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("RESUMEN FINAL DEL CICLO")
print("=" * 60)
print(f"Duración del ciclo:      {t_ciclo_dias} días (~{t_ciclo_dias/30:.0f} meses)")
print(f"Burnup EOC:              {BU[-1]:.0f} MWd/tU")
print(f"U-235 residual al EOC:   {N_U235[-1]*100:.1f}% del inicial")
print(f"k_eff BOC:               {keff_ciclo[0]:.4f}")
print(f"k_eff EOC:               {keff_ciclo[-1]:.4f}")
print(f"Pico de Xe (parada):     {Xe[np.argmax(Xe)]:.2f}× el equilibrio")
print("=" * 60)